# Data merge and feature engineering


In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [3]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [6]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
473880,2024-05-26 20:00:00,2,49.23360,28.44860,18.0,4.1,44.7,0.000,0.0,0.00,0.0,0.0,31.0,12.2,68.2,1023.1,18.7,330.9,28.5,9.0,0.60,20.1,20.1,35.09,0.0,0.0,0.0,23.8,10.4,68.3,1022.0,83.7,121.0,0.4,1.0,True,False,False,False,False,True,False,False,2024,5,6,20,Vinnytsia,Europe/Kiev,2024-05-26,05:10:18,20:57:07,20:00:00,Вінницька,0,0.000000
402048,2024-01-23 02:00:00,2,49.23360,28.44860,-1.8,-4.6,81.1,0.100,100.0,4.17,0.0,7.1,44.3,21.6,191.0,1025.7,98.1,39.9,3.4,2.0,0.42,-2.1,-8.2,88.13,0.0,0.0,0.0,44.3,21.6,180.0,1027.1,100.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2024,1,1,2,Vinnytsia,Europe/Kiev,2024-01-23,07:49:21,16:47:01,02:00:00,Вінницька,0,0.000000
76112,2022-07-06 04:00:00,10,50.45060,30.52430,26.8,15.6,54.8,8.000,100.0,4.17,0.0,0.0,40.3,16.9,185.6,1011.8,62.7,218.5,18.7,8.0,0.23,22.0,22.0,70.09,0.0,0.0,0.0,11.2,4.0,106.8,1014.0,31.3,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2022,7,2,4,Kyiv,Europe/Kiev,2022-07-06,04:54:19,21:10:38,04:00:00,Київська,0,0.000000
574430,2024-11-17 09:00:00,17,50.61930,26.25130,3.4,0.2,80.3,0.000,0.0,0.00,0.0,0.0,31.7,21.4,230.4,1010.7,33.9,0.0,0.0,0.0,0.55,2.1,-0.1,84.74,0.0,0.0,0.0,12.6,7.2,211.6,1012.0,23.4,5.6,0.1,0.0,False,True,False,False,False,True,False,False,2024,11,6,9,Rivne,Europe/Kiev,2024-11-17,07:33:48,16:25:48,09:00:00,Рівненська,1,23.500000
615111,2025-01-27 00:00:00,18,50.90800,34.79720,4.4,3.6,94.6,0.115,100.0,8.33,0.0,0.0,39.6,23.4,193.8,1020.6,100.0,14.4,1.4,1.0,0.93,3.3,-1.4,95.84,0.0,0.0,0.0,38.2,22.0,189.2,1023.0,100.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2025,1,0,0,Sumy,Europe/Kiev,2025-01-27,07:24:12,16:23:41,00:00:00,Сумська,1,60.000000
104596,2022-08-24 15:00:00,6,50.25360,28.66540,23.9,12.3,52.0,0.000,0.0,0.00,0.0,0.0,37.4,19.4,96.4,1018.6,20.7,254.0,21.9,8.0,0.88,31.2,29.5,25.42,0.0,0.0,0.0,36.7,10.8,150.0,1018.1,0.0,711.0,2.6,7.0,False,True,False,False,True,False,False,False,2022,8,2,15,Zhytomyr,Europe/Kiev,2022-08-24,06:07:32,20:06:59,15:00:00,Житомирська,1,13.933333
626308,2025-02-15 11:00:00,6,50.25360,28.66540,-5.8,-9.4,76.9,1.000,100.0,4.17,0.9,1.1,40.3,20.2,320.6,1019.4,83.2,66.4,5.7,3.0,0.59,-5.4,-10.0,62.68,0.0,0.0,0.0,38.2,10.8,340.0,1020.5,40.0,224.7,0.8,2.0,False,False,False,True,False,True,False,False,2025,2,5,11,Zhytomyr,Europe/Kiev,2025-02-15,07:16:18,17:23:29,11:00:00,Житомирська,0,0.000000
551254,2024-10-08 03:00:00,25,51.49370,31.29440,10.6,7.4,81.4,0.100,100.0,4.17,0.0,0.0,52.6,27.4,284.1,1011.5,76.4,57.2,4.9,3.0,0.19,9.2,5.9,92.83,0.0,0.0,0.0,43.9,24.8,281.0,1009.0,100.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2024,10,1,3,Chernihiv,Europe/Kiev,2024-10-08,07:07:32,18:16:11,03:00:00,Чернігівська,0,0.000000
523852,2024-08-21 14:00:00,6,50.25360,28.66540,25.3,13.1,53.1,0.000,0.0,0.00,0.0,0.0,18.0,10.1,120.2,1009.5,15.4,244.9,21.0,8.0,0.56,33.7,32.2,26.08,0.0,0.0,0.0,18.0,7.9,128.8,1009.0,1.4,730.4,2.6,7.0,True,False,False,False,True,False,False,False,2024,8,2,14,Zhytomyr,Europe/Kiev,2024-08-21,06:03:49,20:12:02,14:00:00,Житомирська,0,0.000000
447440,2024-04-10 22:00:00,10,50.45060,30.52430,19.0,6.7,47.3,0.000,0.0,0.00,0.0,0.0,27.4,14.4,206.7,10

In [7]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.4,89.80,0.0,0.0,0.0,16.6,8.3,289.2,1021.0,97.9,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.0,-0.3,90.44,0.0,0.0,0.0,27.7,7.6,287.6,1021.0,98.2,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.9,-0.3,91.75,0.0,0.0,0.0,15.1,7.2,288.9,1021.0,98.8,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.8,-0.2,92.40,0.0,0.0,0.0,13.7,6.5,290.4,1021.0,100.0,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [8]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[ns]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_s

In [9]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
135828,2022-10-17 20:00:00,15,46.4725,30.7371,12.5,5.5,63.3,0.000,0.0,0.00,0.0,0.0,23.8,12.2,28.4,1032.9,1.3,153.9,13.3,6.0,0.71,14.4,14.4,56.65,0.0,0.0,0.0,10.4,5.4,16.3,1034.0,0.0,0.0,0.1,0.0,True,False,False,False,True,False,False,False,2022,10,0,20,Odesa,Europe/Kiev,2022-10-17,07:16:52,18:07:17,20:00:00,Одеська,0,0.0,0.0,1.0,1.0,1.0,12.0,0,0,3.0
562855,2024-10-28 07:00:00,9,48.9226,24.7147,9.8,7.1,84.2,0.000,0.0,0.00,0.0,0.0,30.2,18.0,300.8,1025.0,67.8,129.4,11.2,5.0,0.86,12.3,12.3,81.91,0.0,0.0,0.0,28.1,14.0,314.0,1024.0,75.2,5.6,0.1,0.0,False,True,False,False,False,True,False,False,2024,10,0,7,Ivano-Frankivsk,Europe/Kiev,2024-10-28,07:02:21,17:06:46,07:00:00,Івано-Франківська,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,12.0
214583,2023-03-03 13:00:00,26,50.4506,30.5243,2.2,-1.8,75.4,0.100,100.0,4.17,0.0,0.0,36.7,19.1,267.3,1014.0,85.8,62.4,5.4,3.0,0.37,4.3,0.3,61.73,0.0,0.0,0.0,34.6,19.1,266.0,1012.0,97.4,256.2,0.9,3.0,False,False,True,False,False,True,False,False,2023,3,4,13,Kyiv,Europe/Kiev,2023-03-03,06:38:28,17:42:15,13:00:00,Київ,0,0.0,0.0,1.0,0.0,0.0,4.0,0,0,0.0
238454,2023-04-14 01:00:00,17,50.6193,26.2513,9.3,7.0,86.4,0.007,100.0,8.33,0.0,0.0,40.0,28.5,96.3,1010.4,90.1,118.6,10.4,4.0,0.79,7.7,7.1,93.39,0.0,0.0,0.0,9.4,5.0,100.8,1009.0,22.4,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2023,4,4,1,Rivne,Europe/Kiev,2023-04-14,06:23:58,20:07:51,01:00:00,Рівненська,0,0.0,0.0,0.0,0.0,0.0,2.0,0,1,0.0
358418,2023-11-08 08:00:00,4,48.4753,35.0160,10.3,7.5,83.2,1.000,100.0,4.17,0.0,0.0,46.8,23.4,184.8,1014.7,80.0,33.0,2.8,2.0,0.84,8.2,5.7,78.04,0.0,0.0,0.0,24.5,14.4,190.0,1019.3,90.0,65.4,0.2,1.0,False,False,True,False,False,True,False,False,2023,11,2,8,Dnipro,Europe/Kiev,2023-11-08,06:36:12,16:10:30,08:00:00,Дніпропетровська,0,0.0,0.0,0.0,0.0,0.0,14.0,0,0,0.0
552107,2024-10-09 15:00:00,14,46.9734,31.9852,16.3,12.4,79.6,0.000,0.0,0.00,0.0,0.0,35.6,18.4,171.7,1014.1,48.1,125.6,10.9,4.0,0.22,23.4,23.4,51.69,0.0,0.0,0.0,35.6,10.8,230.0,1012.9,90.0,383.6,1.4,4.0,False,True,False,False,False,True,False,False,2024,10,2,15,Mykolaiv,Europe/Kiev,2024-10-09,07:02:05,18:15:41,15:00:00,Миколаївська,0,0.0,0.0,0.0,0.0,1.0,9.0,0,0,0.0
540295,2024-09-19 03:00:00,9,48.9226,24.7147,15.1,9.4,72.2,0.900,100.0,4.17,0.0,0.0,28.8,14.4,91.5,1025.5,62.4,142.2,12.2,6.0,0.54,13.2,13.2,91.83,0.0,0.0,0.0,21.2,3.6,70.0,1025.4,60.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2024,9,3,3,Ivano-Frankivsk,Europe/Kiev,2024-09-19,07:03:47,19:25:00,03:00:00,Івано-Франківська,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,11.0
482234,2024-06-10 08:00:00,4,48.4753,35.0160,25.8,13.2,49.6,0.000,0.0,0.00,0.0,0.0,37.1,21.6,188.7,1011.0,45.4,292.9,25.4,8.0,0.12,26.4,26.4,50.82,0.0,0.0,0.0,19.8,9.0,168.6,1012.0,95.1,326.2,1.2,3.0,False,True,False,False,False,True,False,False,2024,6,0,8,Dnipro,Europe/Kiev,2024-06-10,04:38:28,20:40:55,08:00:00,Дніпропетровська,0,0.0,0.0,0.0,0.0,0.0,10.0,0,0,0.0
250745,2023-05-05 09:00:00,20,50.0042,36.2358,14.4,6.4,60.0,0.300,100.0,8.33,0.0,0.0,31.3,21.6,358.2,1018.6,84.9,221.2,19.1,7.0,0.50,17.1

In [10]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [11]:
def calculate_hours_since_last(series):
    series = series.shift(1).fillna(0)
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last))

In [12]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
22931,2024-10-06 14:00:00,2,49.23360,28.44860,15.3,14.2,93.2,17.300,100.0,8.33,0.0,0.0,26.3,11.5,282.4,1008.4,100.0,39.3,3.5,2.0,0.12,15.6,15.6,89.62,0.000,0.0,0.0,22.7,9.4,279.9,1008.0,100.0,100.6,0.4,1.0,False,False,True,False,False,True,False,False,2024,10,6,14,Vinnytsia,Europe/Kiev,2024-10-06,07:13:47,18:33:53,14:00:00,Вінницька,0,0.000000,0.0,0.0,0.0,1.0,10.0,1,0,2.0,0.0,3
111284,2022-10-11 09:00:00,6,50.25360,28.66540,7.4,2.2,73.5,0.000,0.0,0.00,0.0,0.0,20.2,10.1,210.0,1024.2,27.7,143.4,12.4,5.0,0.51,4.2,4.2,89.96,0.000,0.0,0.0,5.0,0.0,188.6,1025.4,10.0,116.0,0.4,1.0,False,True,False,False,True,False,False,False,2022,10,1,9,Zhytomyr,Europe/Kiev,2022-10-11,07:20:47,18:22:37,09:00:00,Житомирська,1,60.000000,1.0,0.0,0.0,0.0,6.0,0,0,24.0,4.0,0
264886,2022-03-14 04:00:00,13,49.84440,24.02540,1.8,-5.1,62.7,0.000,0.0,0.00,0.0,0.0,28.1,14.4,143.8,1034.5,0.0,191.4,16.5,5.0,0.38,-2.5,-5.1,79.85,0.000,0.0,0.0,14.8,6.5,112.5,1035.0,0.0,0.0,0.1,0.0,True,False,False,False,True,False,False,False,2022,3,0,4,Lviv,Europe/Kiev,2022-03-14,06:40:02,18:27:07,04:00:00,Львівська,1,60.000000,1.0,0.0,0.0,0.0,6.0,0,1,19.0,5.0,0
365452,2024-08-14 22:00:00,16,49.58790,34.55170,18.5,9.6,58.9,0.000,0.0,0.00,0.0,0.0,38.9,18.0,319.1,1016.5,25.5,193.2,16.6,8.0,0.32,19.5,19.5,54.91,0.000,0.0,0.0,12.6,7.6,326.0,1018.0,1.5,0.0,0.0,0.0,False,True,False,False,True,False,False,False,2024,8,2,22,Poltava,Europe/Kiev,2024-08-14,05:31:32,20:00:24,22:00:00,Полтавська,1,0.600000,1.0,1.0,0.0,1.0,14.0,0,0,7.0,4.0,0
232253,2024-07-05 08:00:00,10,50.45060,30.52430,20.8,14.6,70.0,0.700,100.0,12.50,0.0,0.0,39.6,19.4,300.1,1009.7,59.8,270.3,23.5,8.0,0.98,18.7,18.7,86.49,0.100,100.0,0.0,27.0,13.3,294.0,1009.0,82.4,155.5,0.6,2.0,False,False,True,False,False,False,True,False,2024,7,4,8,Kyiv,Europe/Kiev,2024-07-05,04:53:57,21:10:54,08:00:00,Київська,0,0.000000,0.0,0.0,1.0,0.0,6.0,0,0,7.0,3.0,3
121252,2023-11-30 18:00:00,6,50.25360,28.66540,-3.8,-7.4,76.9,0.000,0.0,0.00,0.0,6.2,45.4,22.0,227.8,1011.6,65.4,56.9,4.9,3.0,0.60,-2.1,-7.5,71.83,0.000,0.0,0.0,28.1,17.3,198.4,1012.0,20.8,0.0,0.0,0.0,False,True,False,False,False,True,False,False,2023,11,3,18,Zhytomyr,Europe/Kiev,2023-11-30,07:41:19,16:06:10,18:00:00,Житомирська,0,0.000000,0.0,0.0,0.0,0.0,2.0,0,0,4.0,0.0,12
139667,2022-12-31 03:00:00,7,48.62636,22.28514,6.4,5.6,95.1,1.088,100.0,8.33,0.0,0.4,28.8,10.8,155.6,1025.7,98.0,8.3,0.6,0.0,0.25,5.0,3.3,95.63,0.000,0.0,0.0,17.6,7.2,160.0,1024.6,100.0,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2022,12,5,3,Uzhgorod,Europe/Uzhgorod,2022-12-31,08:23:27,16:44:23,03:00:00,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,1.0,1,1,1.0,0.0,14
255817,2024-03-07 06:00:00,11,48.50850,32.26560,-0.5,-5.0,72.2,0.000,0.0,0.00,0.0,0.0,29.9,18.0,32.9,1023.8,93.9,81.0,6.9,3.0,0.90,-2.0,-6.6,81.76,0.000,0.0,0.0,26.3,13.7,36.5,1023.0,94.9,0.0,0.0,0.0,False,True,False,False,False,True,False,False,2024,3,3,6,Kropyvnytskyi,Europe/Kiev,2024-03-07,06:19:58,17:44:38,06:00:00,Кіровоградська,0,0.000000,0.0,0.0,0.0,0.0,6.0

In [13]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
528,2023-08-09,0.708288,-0.211903,-0.086730,-0.149092,-0.121213,0.150973,-0.031438,0.208107,0.025112,0.037309,0.165840,0.028634,-0.188834,-0.065981,0.060041,-0.054538,0.099998,-0.110803,0.038449,-0.045920,0.141840,0.015058,0.042988,0.004215,0.083478,0.093383,0.014700,-0.003323,-0.018350,0.003488,-0.040702,-0.004049,-0.073432,-0.063260,-0.024153,0.028816,-0.024412,0.050774,0.033323,-0.069316,-0.049126,-0.017278,0.016653,-0.004501,-0.024927,-0.047902,-0.021401,0.023503,-0.028247,-0.012019,-0.033458,-0.045759,-0.005521,-0.004911,0.031088,0.011968,-0.007987,0.036255,0.016708,0.007343,0.010055,0.070523,-0.016265,0.018375,-0.003569,0.034085,0.014384,-0.008675,-0.017706,0.008202,-0.015770,0.055711,-0.029004,-0.006545,-0.030657,0.041676,0.014917,-0.000847,-0.000201,0.004474,-0.009375,0.007897,-0.009739,-0.016868,-0.007504,0.004947,0.022153,0.007445,0.007150,0.038039,-0.000751,0.010839,-0.030582,-0.016455,-0.045452,0.037182,-0.018901,0.041864,0.006250,-0.051237,0.003211,-0.026736,0.000048,-0.014140,0.002200,-0.020378,-0.055255,-0.008109,0.017916,-0.004029,0.025969,0.018326,0.028489,0.000817,-0.033030,-0.018640,0.029765,-0.011588,0.022258,0.012079,-0.020116,0.006592,0.006224,-0.000510,0.004841,0.034687,0.053619,0.008425,-0.047183,0.008075,-0.001763,0.033182,-0.000492,-0.013596,-0.011538,0.032250,-0.007786,-0.005489,-0.023369,0.007861,-0.015272,0.024549,-0.009216,-0.006258,-0.002414,-0.016496,0.043290,0.003357,-0.018537,-0.034981
1209,2025-06-27,0.786359,0.271797,0.035545,0.084521,-0.126942,0.069325,0.053996,-0.048383,-0.102854,0.078990,-0.024602,0.250191,-0.064038,0.036933,-0.062007,0.023902,-0.044686,-0.035940,0.035241,-0.009808,-0.082374,-0.011506,0.033467,-0.031841,-0.048449,0.025298,-0.049090,0.012376,0.038031,0.047115,0.021643,-0.032082,-0.006635,-0.022580,0.002990,-0.005210,0.023248,0.041557,-0.020761,-0.009079,-0.002803,-0.015733,0.028827,-0.001799,-0.023302,-0.025874,0.043065,-0.000529,-0.021737,-0.047330,-0.018417,-0.019992,0.041693,0.019244,-0.031595,0.006256,0.028083,-

In [14]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [15]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [16]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [17]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
100347,2024-07-18 15:00:00,5,48.0020,37.8145,30.5,15.0,41.4,0.000,0.0,0.00,0.0,0.0,29.9,11.5,27.1,1012.6,16.2,280.2,24.3,7.0,0.40,36.7,35.9,24.70,0.0,0.0,0.0,24.1,8.6,38.1,1012.0,35.2,633.3,2.3,6.0,True,False,False,False,False,True,False,False,2024,7,3,15,Donetsk,Europe/Kiev,2024-07-18,04:48:28,20:21:04,15:00:00,Донецька,1,58.050000,0.0,1.0,1.0,1.0,22.0,0,0,8.0,3.0,2,0.760267,0.015178,-0.078852,0.070842,-0.200958,0.139618,0.109170,-0.137824,-0.018277,0.071013,-0.174145,-0.277872,0.069108,-0.017035,-0.031844,-0.011416,0.013544,-0.077641,0.029381,0.029859,-0.024153,-0.064072,0.043107,0.051210,0.006818,-0.017748,0.046095,-0.043080,-0.023213,0.019518,-0.022270,-0.043839,0.021073,-0.030306,-0.023382,-0.022829,-0.018568,-0.023327,0.043550,0.042733,0.031799,-0.019315,0.033144,0.037504,-0.018285,0.032516,0.025820,0.026405,0.025765,0.007843,0.031287,-0.040047,0.002266,-0.005384,0.016674,-0.014792,-0.006102,0.033531,0.000010,0.006225,-0.058988,0.005989,-0.007321,0.043381,0.013355,-0.019772,0.007989,0.0

In [18]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     6336
isw_topic_146     6336
isw_topic_147     6336
isw_topic_148     6336
isw_topic_149     6336
Length: 216, dtype: int64

In [19]:
df_final.fillna(0, inplace=True)

In [20]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [21]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

In [22]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
235691,2024-11-25 14:00:00,10,50.4506,30.5243,-2.8,-4.1,91.0,0.0,0.0,0.00,0.0,3.5,23.0,11.5,148.2,1033.0,72.5,41.5,3.6,2.0,0.81,2.1,2.1,75.38,0.0,0.0,0.0,19.8,3.6,130.0,1032.9,99.2,164.8,0.6,2.0,False,True,False,False,False,True,False,False,2024,11,0,14,Kyiv,Europe/Kiev,2024-11-25,07:28:36,16:01:06,14:00:00,Київська,0,0.0,0.0,0.0,1.0,1.0,17.0,0,0,3.0,0.0,4,0.809575,0.126041,0.006566,-0.052762,-0.010295,-0.148950,0.073070,-0.145940,0.172992,0.089046,0.014239,0.024531,-0.025037,0.030282,0.159686,0.015215,-0.005100,-0.008028,0.000373,-0.007137,0.033925,0.014047,-0.055206,-0.063304,-0.019147,0.026678,-0.046679,0.056787,0.033246,0.035540,-0.053926,0.038427,0.001441,-0.036659,-0.037144,-0.037759,0.008293,0.031385,-0.017946,0.017923,0.001561,-0.007191,0.003230,0.029285,0.008432,-0.035362,0.056134,-0.069675,-0.033101,0.004075,-0.027206,0.037080,-0.040772,0.051121,0.038028,-0.014109,0.00

### TELEGRAM MERGE

In [23]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [24]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000991,-0.011395,0.032671,-0.050983,0.061079,0.042624,-0.024342,-0.018764,0.023401,-0.017136,-0.358835,0.425288,0.017963,-0.107665,-0.032070,-0.045988,0.118212,0.091173,-0.248636,-0.001072,0.150764,0.030607,-0.074025,0.005802,0.005872,0.034871,0.017239,0.058098,0.029814,-0.010846,0.031485,-0.022706,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005138,0.007857,0.016733,-0.040291,-0.019686,-0.026762,0.009384,-0.043499,-0.009524,-0.013075,-0.010474,-0.018151,0.005990,0.032610,0.000850,0.017870,0.020887,0.023896,-0.019292,-0.006676,-0.018478,-0.023265,-0.005496,-0.006460,0.026051,0.004991,0.019690,-0.022244,0.010931,0.007016,-0.067334,-0.035279,-0.005691,0.018746,0.039153,-0.053869,-0.078182,-0.074734,-0.082173,0.020503,0.084333,0.113472,-0.025870,-0.048315,0.096

In [25]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

In [26]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .ffill()
    .reset_index()
)

In [27]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000991,-0.011395,0.032671,-0.050983,0.061079,0.042624,-0.024342,-0.018764,0.023401,-0.017136,-0.358835,0.425288,0.017963,-0.107665,-0.032070,-0.045988,0.118212,0.091173,-0.248636,-0.001072,0.150764,0.030607,-0.074025,0.005802,0.005872,0.034871,0.017239,0.058098,0.029814,-0.010846,0.031485,-0.022706,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005138,0.007857,0.016733,-0.040291,-0.019686,-0.026762,0.009384,-0.043499,-0.009524,-0.013075,-0.010474,-0.018151,0.005990,0.032610,0.000850,0.017870,0.020887,0.023896,-0.019292,-0.006676,-0.018478,-0.023265,-0.005496,-0.006460,0.026051,0.004991,0.019690,-0.022244,0.010931,0.007016,-0.067334,-0.035279,-0.005691,0.018746,0.039153,-0.053869,-0.078182,-0.074734,-0.082173,0.020503,0.084333,0.113472,-0.025870,-

In [28]:
tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)

In [29]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [30]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [31]:
df_final.shape

(634680, 473)

In [32]:
df_final = df_final.sort_values(["datetime_hour"])

In [33]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
city_latitude                         0
city_longitude                        0
day_temp                              0
day_dew                               0
day_humidity                          0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_snow                              0
day_snowdepth                         0
day_windgust                          0
day_windspeed                         0
day_winddir                           0
day_pressure                          0
day_cloudcover                        0
day_solarradiation                    0
day_solarenergy                       0
day_uvindex                           0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_precip                           0


In [34]:
df_final = df_final.fillna(0)

In [35]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

In [36]:
df_final.head(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [38]:
df_final.to_parquet("merged_dataset.parquet", index=False, engine="pyarrow")